# EDA — RetailTech S.A.S

Análisis exploratorio de los datos fuente para identificar problemas de calidad y patrones.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
RAW = Path('..') / 'data' / 'raw'

# Cargar datasets
clientes = pd.read_csv(RAW / 'clientes.csv')
productos = pd.read_csv(RAW / 'productos.csv')
pedidos = pd.read_csv(RAW / 'pedidos.csv')
detalle = pd.read_csv(RAW / 'detalle_pedidos.csv')
eventos = pd.read_csv(RAW / 'eventos.csv')
diccionario = pd.read_csv(RAW / 'diccionario_datos.csv')

print('Datasets cargados:')
for name, df in [('clientes', clientes), ('productos', productos), 
                  ('pedidos', pedidos), ('detalle', detalle), ('eventos', eventos)]:
    print(f'  {name}: {df.shape[0]} filas x {df.shape[1]} columnas')

## 1. Clientes

In [ ]:
print('=== CLIENTES ===')
print(f'Filas: {len(clientes)}')
print(f'\nNulos por columna:')
nulls = clientes.isnull().sum()
print(nulls[nulls > 0])

print(f'\nTeléfonos N/A: {(clientes["telefono"] == "N/A").sum()}')
print(f'Ciudades vacías: {clientes["ciudad"].isna().sum() + (clientes["ciudad"].astype(str).str.strip() == "").sum()}')
print(f'\nPaíses: {clientes["pais"].value_counts().to_dict()}')
print(f'Segmentos: {clientes["segmento"].value_counts().to_dict()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
clientes['pais'].value_counts().plot.bar(ax=axes[0], color='#3483FA')
axes[0].set_title('Clientes por País')
axes[0].tick_params(axis='x', rotation=45)

clientes['segmento'].value_counts().plot.bar(ax=axes[1], color='#FFE600', edgecolor='#333')
axes[1].set_title('Clientes por Segmento')
plt.tight_layout()
plt.show()

## 2. Productos

In [ ]:
print('=== PRODUCTOS ===')
print(f'Filas: {len(productos)}')
print(f'\nNulos:')
nulls = productos.isnull().sum()
print(nulls[nulls > 0])
print(f'\nCategorías: {productos["categoria"].value_counts().to_dict()}')
print(f'\nPrecio venta: min={productos["precio_venta"].min():.0f}, max={productos["precio_venta"].max():.0f}, mean={productos["precio_venta"].mean():.0f}')
print(f'Productos con precio_venta <= costo: {(productos["precio_venta"] <= productos["costo"]).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
productos['categoria'].value_counts().plot.bar(ax=axes[0], color='#3483FA')
axes[0].set_title('Productos por Categoría')
axes[0].tick_params(axis='x', rotation=45)

axes[1].hist(productos['precio_venta'].dropna(), bins=20, color='#FFE600', edgecolor='#333')
axes[1].set_title('Distribución de Precios de Venta')
axes[1].set_xlabel('Precio (COP)')
plt.tight_layout()
plt.show()

## 3. Pedidos

In [ ]:
print('=== PEDIDOS ===')
print(f'Filas: {len(pedidos)}')
print(f'pedido_id duplicados: {pedidos["pedido_id"].duplicated().sum()}')
print(f'\nNulos:')
nulls = pedidos.isnull().sum()
print(nulls[nulls > 0])

print(f'\nEstados: {pedidos["estado"].value_counts().to_dict()}')
print(f'Canales: {pedidos["canal"].value_counts().to_dict()}')
print(f'Países: {pedidos["pais_envio"].value_counts().to_dict()}')

# Validar FK
clientes_ids = set(clientes['cliente_id'])
fk_missing = pedidos[~pedidos['cliente_id'].isin(clientes_ids)]
print(f'\nPedidos con cliente_id no existente: {len(fk_missing)}')

In [ ]:
pedidos['fecha_pedido'] = pd.to_datetime(pedidos['fecha_pedido'])
monthly = pedidos.groupby(pedidos['fecha_pedido'].dt.to_period('M')).agg(
    total=('pedido_id', 'count'),
    revenue=('total_neto', 'sum')
).reset_index()
monthly['fecha_pedido'] = monthly['fecha_pedido'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(monthly['fecha_pedido'], monthly['total'], marker='o', color='#3483FA')
axes[0].set_title('Pedidos por Mes')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(monthly['fecha_pedido'], monthly['revenue'], color='#FFE600', edgecolor='#333')
axes[1].set_title('Revenue Neto Mensual')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 4. Detalle Pedidos

In [ ]:
print('=== DETALLE PEDIDOS ===')
print(f'Filas: {len(detalle)}')
print(f'\nNulos:')
nulls = detalle.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else 'Sin nulos')
print(f'\nProductos únicos referenciados: {detalle["producto_id"].nunique()}')
print(f'Pedidos únicos referenciados: {detalle["pedido_id"].nunique()}')

## 5. Eventos

In [ ]:
print('=== EVENTOS ===')
print(f'Filas: {len(eventos)}')
print(f'\nNulos:')
nulls = eventos.isnull().sum()
print(nulls[nulls > 0])
print(f'\ncliente_id nulos (anónimos): {eventos["cliente_id"].isna().sum()} ({eventos["cliente_id"].isna().mean()*100:.1f}%)')
print(f'duracion_seg nulos: {eventos["duracion_seg"].isna().sum()}')
print(f'\nTipos de evento: {eventos["tipo_evento"].value_counts().to_dict()}')
print(f'Dispositivos: {eventos["dispositivo"].value_counts().to_dict()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
eventos['tipo_evento'].value_counts().plot.bar(ax=axes[0], color='#3483FA')
axes[0].set_title('Eventos por Tipo')
axes[0].tick_params(axis='x', rotation=45)

eventos['dispositivo'].value_counts().plot.pie(ax=axes[1], autopct='%1.0f%%', colors=['#FFE600', '#3483FA', '#2D3277'])
axes[1].set_title('Eventos por Dispositivo')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

## 6. Resumen de Hallazgos

| # | Tabla | Problema | Registros |
|---|-------|----------|-----------|
| 1 | clientes | ~5% emails nulos | ~15 |
| 2 | clientes | ~8% teléfonos "N/A" | ~24 |
| 3 | clientes | ~3% ciudades vacías | ~9 |
| 4 | productos | ~5% stock_disponible nulo | ~4 |
| 5 | pedidos | ~2% pedido_id duplicados | ~21 |
| 6 | eventos | ~30% cliente_id nulo (anónimos) | ~1200 |
| 7 | eventos | ~5% duracion_seg nulo | ~200 |

**PII identificado:** nombre, apellido, email, telefono, fecha_consentimiento (clientes)

**Acciones:**
- Bronze: documentar estado as-is
- Silver: aplicar correcciones + masking PII
- Gold: agregar para analytics